# Classification Probability Layers

Ensemble posterior、calibration、reliability、conformalの責務をprobability-spaceで整理します。


In [ ]:
import torch

from robotorchan.models.classification import (
    ClassificationEnsemblePosterior,
    TemperatureScalingCalibrator,
    classification_nll,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Probability ensemble posterior

In [ ]:
p = grid.squeeze(-1)
members = torch.stack([
    torch.stack((1.0 - p, p), dim=-1),
    torch.stack((1.0 - p.pow(0.8), p.pow(0.8)), dim=-1),
    torch.stack((1.0 - p.pow(1.2), p.pow(1.2)), dim=-1),
])
posterior = ClassificationEnsemblePosterior(members)
print(posterior.mean.shape)
print(posterior.variance.shape)


## 3. Probability calibration

In [ ]:
raw = torch.tensor([[0.90, 0.10], [0.65, 0.35], [0.30, 0.70], [0.05, 0.95]])
targets = torch.tensor([0, 0, 1, 1])
calibrator = TemperatureScalingCalibrator()
calibrator.fit(raw, targets)
calibrated = calibrator(raw)
print(classification_nll(raw, targets).item())
print(classification_nll(calibrated, targets).item())


## 4. Reliability / conformalとの違い

Reliability/OODはentropy・disagreement・input distanceを分離して診断します。Split conformalはheld-out calibration setからprediction setを構成します。probability calibrationとは別レイヤーです。

実運用ではmodel fitting、probability calibration、conformal calibrationに使うデータの役割を明示してください。

## 5. Probability-space候補選択とラベル評価

このNotebookのensembleは合成確率メンバーであり、学習済み分類器ではありません。Probability calibrationは候補最適化ではなく確率の補正です。ここではensemble disagreementによる候補選択を示します。


In [ ]:
candidate_disagreement = posterior.variance[..., 1]
candidate = grid[candidate_disagreement.argmax()].unsqueeze(0)
observed_label = (candidate.squeeze(-1) > 0.5).long()
print("candidate:", candidate)
print("observed label:", observed_label)


## 初学者向け補足：確率を出した後にも評価が必要

分類器が `0.9` と予測しても、実際に同じような予測を100件集めて約90件がクラス1になるとは限りません。確率の数値と実際の頻度が一致する性質を **calibration（確率校正）** と呼びます。

| 概念 | 答えたい問い |
|---|---|
| Ensemble | 複数モデルの予測はどの程度一致するか |
| Calibration | 予測確率と実際の頻度は整合しているか |
| Reliability / OOD | 未知の入力や信頼しづらい予測を検出できるか |
| Conformal prediction | 仮定のもとで所定の被覆率を目指す予測集合を作れるか |

### このNotebookで注意すること

このNotebookのensembleは **合成した確率メンバー** を用いる説明例です。学習済みGP分類器を複数訓練した性能実験ではありません。

校正を評価するときは、学習・校正・最終評価のデータを適切に分離します。Brier ScoreやLog Lossは確率の質を見る指標ですが、Accuracyとは測っているものが異なります。Conformalの被覆率も、必要な交換可能性などの仮定を満たすか確認してください。

### 読み取りの練習

予測確率が0.5に近い点と、メンバー間の意見が割れる点は必ずしも同じではありません。各セルの出力を見ながら、どの指標がどちらを強調するか考えてみてください。